# Gulf of Tunis — Multi-Source Ocean Data Tutorial

This notebook shows how to access and visualize ocean observations over the **Gulf of Tunis** from two cloud-native data services:

| # | Variable | Sensor / Product | Service |
|---|----------|------------------|---------|
| 1 | Sea Surface Temperature (gridded, L3) | CMEMS SST MED NRT | Copernicus Marine |
| 2 | Sea Surface Temperature (swath, L2) | Sentinel-3A SLSTR | Planetary Computer (STAC) |
| 3 | Chlorophyll-a (swath, L2) | Sentinel-3A OLCI | Planetary Computer (STAC) |

**Learning objectives:**
- Use the `copernicusmarine` Python client to stream gridded satellite data
- Use `pystac_client` to search a STAC API and open assets with `xarray`
- Clip, mask, and visualize 2-D swath data over a small area of interest
- Compare SST from two independent sources over the same region

## 0. Shared setup

We define the **bounding box** once and reuse it throughout. All three sections clip their data to this box.

In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import xarray as xr
import fsspec
import hvplot.xarray
import hvplot.pandas

# Area of interest: Gulf of Tunis  [lon_min, lat_min, lon_max, lat_max]
bbox = [10.0, 36.6, 10.9, 37.2]
print(f"Gulf of Tunis bbox: lon [{bbox[0]}, {bbox[2]}]  lat [{bbox[1]}, {bbox[3]}]")

Gulf of Tunis bbox: lon [10.0, 10.9]  lat [36.6, 37.2]


---
## 1. CMEMS gridded SST (Copernicus Marine Service)

The **Copernicus Marine Service** (CMEMS) distributes processed, gridded satellite products via an OPeNDAP/Zarr backend. The `copernicusmarine` Python client lets us stream only the data we need — no download required.

We use:
- **Dataset:** `SST_MED_SST_L3S_NRT_OBSERVATIONS_010_012_b`
- **Variable:** `adjusted_sea_surface_temperature` (Kelvin → °C)
- **Grid:** ~0.01° regular lat/lon over the Mediterranean

> **Credentials:** the `~/.netrc` file must contain your Copernicus Marine username and password.

In [2]:
import copernicusmarine

dataset_id = "SST_MED_SST_L3S_NRT_OBSERVATIONS_010_012_b"

ds_cmems = copernicusmarine.open_dataset(dataset_id)
print(list(ds_cmems.data_vars))

INFO - 2026-04-22T13:21:19Z - Downloading Copernicus Marine data requires a Copernicus Marine username and password, sign up for free at: https://data.marine.copernicus.eu/register


Copernicus Marine username:

  gmozzi


Copernicus Marine password:

  ········


INFO - 2026-04-22T13:21:30Z - Selected dataset version: "202311"
INFO - 2026-04-22T13:21:30Z - Selected dataset part: "default"
WARNING - 2026-04-22T13:21:30Z - The dataset SST_MED_SST_L3S_NRT_OBSERVATIONS_010_012_b, version '202311', part 'default' is currently being updated. Data after 2026-04-22T00:00:00.000Z may not be up to date.


[np.str_('adjusted_sea_surface_temperature'), np.str_('adjusted_standard_deviation_error'), np.str_('bias_to_reference_sst'), np.str_('l2p_flags'), np.str_('or_latitude'), np.str_('or_longitude'), np.str_('or_number_of_pixels'), np.str_('quality_level'), np.str_('sea_surface_temperature'), np.str_('sources_of_sst'), np.str_('sses_bias'), np.str_('sses_standard_deviation'), np.str_('sst_dtime'), np.str_('standard_deviation_to_reference_sst'), np.str_('sum_square_sst'), np.str_('sum_sst'), np.str_('wind_speed')]


Select a single time step and clip to the Gulf of Tunis bbox using `.sel()`.

> If the latitude slice returns empty data, the axis may be stored in descending order — swap the limits: `slice(bbox[3], bbox[1])`.

In [8]:
time_val="2026-01-01 16:00"
da_sst_cmems = (
    ds_cmems["adjusted_sea_surface_temperature"]
    .sel(time=time_val, method="nearest")
    .sel(
        longitude=slice(bbox[0], bbox[2]),
        latitude=slice(bbox[1], bbox[3]),
    )
    .load()
) - 273.15  # Kelvin → Celsius

print(f"Shape: {da_sst_cmems.shape}")
print(f"SST range: {float(da_sst_cmems.min()):.1f} – {float(da_sst_cmems.max()):.1f} °C")

Shape: (72, 108)
SST range: 16.5 – 17.9 °C


In [11]:
da_sst_cmems = (
    ds_cmems["adjusted_sea_surface_temperature"]
    .sel(
        longitude=slice(bbox[0], bbox[2]),
        latitude=slice(bbox[1], bbox[3]),
    )
) - 273.15  # Kelvin → Celsius

print(f"Shape: {da_sst_cmems.shape}")
print(f"SST range: {float(da_sst_cmems.min()):.1f} – {float(da_sst_cmems.max()):.1f} °C")

Shape: (6687, 72, 108)


KeyboardInterrupt: 

In [12]:
da_sst_cmems.hvplot(
    x="longitude", y="latitude",
    rasterize=True,
    geo=True,
    cmap="RdYlBu_r",
    clabel="SST (°C)",
    #title=f"CMEMS Gridded SST — Gulf of Tunis {time_val}",
    tiles="OSM",
    frame_width=700,
)

:DynamicMap   [time]
   :Overlay
      .WMTS.I  :WMTS   [Longitude,Latitude]
      .Image.I :Image   [longitude,latitude]   (adjusted_sea_surface_temperature)

---
## 2. Sentinel-3 SLSTR SST — swath data via Planetary Computer (STAC)

**Planetary Computer** hosts a STAC (SpatioTemporal Asset Catalog) API that indexes individual satellite passes. Unlike gridded products, these are raw Level-2 **swath** files: irregular grids that follow the satellite track.

We use:
- **Collection:** `sentinel-3-slstr-wst-l2-netcdf`
- **Variable:** `sea_surface_temperature` (Kelvin → °C)
- **Resolution:** ~1 km at nadir

**Workflow:**
1. Connect to the STAC API
2. Search by bbox + date range
3. Sign the asset URL (Planetary Computer requires this)
4. Open directly with `xarray` via `fsspec` — no download
5. Mask pixels to the exact bbox and visualize

In [ ]:
import pystac_client
import planetary_computer

catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

search_sst = catalog.search(
    collections=["sentinel-3-slstr-wst-l2-netcdf"],
    bbox=bbox,
    datetime="2026-01-01/2026-04-21",
)

items_sst = search_sst.item_collection()
print(f"Found {len(items_sst)} Sentinel-3 SLSTR passes")
print(f"First pass: {items_sst[0].datetime}")

In [ ]:
item_sst = items_sst[0]

asset = planetary_computer.sign(item_sst.assets["l2p"])
ds_sst = xr.open_dataset(fsspec.open(asset.href).open(), decode_timedelta=False)

print(f"Dimensions: {dict(ds_sst.dims)}")
print(f"Coordinates: {list(ds_sst.coords)}")

The SLSTR L2P file has a 2-D irregular grid (`nj × ni`). We clip it by building a boolean mask from the `lat`/`lon` arrays, then extract the minimal rectangular bounding slice.

In [ ]:
lat_sst = ds_sst["lat"].values
lon_sst = ds_sst["lon"].values

mask_sst = (
    (lat_sst >= bbox[1]) & (lat_sst <= bbox[3]) &
    (lon_sst >= bbox[0]) & (lon_sst <= bbox[2])
)

nj_idx, ni_idx = np.where(mask_sst)
print(f"{len(nj_idx)} pixels found in the Gulf of Tunis")

nj_min, nj_max = nj_idx.min(), nj_idx.max() + 1
ni_min, ni_max = ni_idx.min(), ni_idx.max() + 1

sst_sub = ds_sst["sea_surface_temperature"].isel(
    nj=slice(nj_min, nj_max), ni=slice(ni_min, ni_max), time=0
).load() - 273.15

lat_sub = lat_sst[nj_min:nj_max, ni_min:ni_max]
lon_sub = lon_sst[nj_min:nj_max, ni_min:ni_max]

# Apply exact bbox mask (remove rectangular envelope pixels outside bbox)
mask_sub = mask_sst[nj_min:nj_max, ni_min:ni_max]
sst_vals = sst_sub.values.copy()
sst_vals[~mask_sub] = np.nan

valid = sst_vals[np.isfinite(sst_vals) & (sst_vals > 10)]
print(f"SST range: {valid.min():.1f} – {valid.max():.1f} °C  |  mean: {valid.mean():.1f} °C")

In [ ]:
df_sst = pd.DataFrame({
    "lat": lat_sub.flatten(),
    "lon": lon_sub.flatten(),
    "sst": sst_vals.flatten(),
})
df_sst = df_sst[(df_sst["sst"] > 10) & (df_sst["sst"] < 40)]

df_sst.hvplot.points(
    x="lon", y="lat",
    c="sst",
    colormap="RdYlBu_r",
    clabel="SST (°C)",
    title=f"Sentinel-3 SLSTR SST — Gulf of Tunis ({str(item_sst.datetime.date())})",
    xlabel="Longitude", ylabel="Latitude",
    size=6,
    geo=True,
    tiles="OSM",
    width=700, height=500,
)

---
## 3. Sentinel-3 OLCI Chlorophyll-a — swath data via Planetary Computer (STAC)

Chlorophyll-a (CHL) is a proxy for **phytoplankton biomass** — a key biological indicator of ocean productivity. We use the **Neural Net** algorithm output (`CHL_NN`) from the OLCI Water Full Resolution (WFR) product.

- **Collection:** `sentinel-3-olci-wfr-l2-netcdf`
- **Variable:** `CHL_NN` (mg/m³) — best for coastal and turbid waters
- **Resolution:** ~300 m

**Important:** the `chl_nn.nc` asset contains only the data values. Lat/lon coordinates live in a separate `geo-coordinates` asset and must be loaded and merged.

In [ ]:
search_chl = catalog.search(
    collections=["sentinel-3-olci-wfr-l2-netcdf"],
    bbox=bbox,
    datetime="2026-01-01/2026-04-21",
)

items_chl = search_chl.item_collection()
print(f"Found {len(items_chl)} Sentinel-3 OLCI passes")
print(f"First pass: {items_chl[0].datetime}")

In [ ]:
def open_asset(item, key):
    """Sign and open a Planetary Computer asset as an xarray Dataset."""
    asset = planetary_computer.sign(item.assets[key])
    return xr.open_dataset(fsspec.open(asset.href).open())

item_chl = items_chl[0]

ds_chl = open_asset(item_chl, "chl-nn")
geo    = open_asset(item_chl, "geo-coordinates")

# Merge lat/lon (stored in a separate file) as coordinates
ds_chl = ds_chl.assign_coords(
    latitude=(["rows", "columns"], geo["latitude"].values),
    longitude=(["rows", "columns"], geo["longitude"].values),
)

print(ds_chl)

Same spatial masking approach as for SST: boolean mask → rectangular slice → re-apply mask.

In [ ]:
lat_chl = ds_chl["latitude"].values
lon_chl = ds_chl["longitude"].values

mask_chl = (
    (lat_chl >= bbox[1]) & (lat_chl <= bbox[3]) &
    (lon_chl >= bbox[0]) & (lon_chl <= bbox[2])
)

rows_idx, cols_idx = np.where(mask_chl)
print(f"{len(rows_idx)} pixels found in the Gulf of Tunis")

row_min, row_max = rows_idx.min(), rows_idx.max() + 1
col_min, col_max = cols_idx.min(), cols_idx.max() + 1

chl_sub = ds_chl["CHL_NN"].isel(
    rows=slice(row_min, row_max),
    columns=slice(col_min, col_max),
).load()

lat_sub_chl = lat_chl[row_min:row_max, col_min:col_max]
lon_sub_chl = lon_chl[row_min:row_max, col_min:col_max]

mask_sub_chl = mask_chl[row_min:row_max, col_min:col_max]
chl_vals = chl_sub.values.copy()
chl_vals[~mask_sub_chl] = np.nan

valid_chl = chl_vals[np.isfinite(chl_vals) & (chl_vals > 0)]
print(f"CHL_NN range: {valid_chl.min():.3f} – {valid_chl.max():.3f} mg/m³  |  mean: {valid_chl.mean():.3f} mg/m³")

In [ ]:
df_chl = pd.DataFrame({
    "lat": lat_sub_chl.flatten(),
    "lon": lon_sub_chl.flatten(),
    "chl": chl_vals.flatten(),
})
df_chl = df_chl[df_chl["chl"].notna() & (df_chl["chl"] > 0) & (df_chl["chl"] < 100)]
df_chl["log_chl"] = np.log10(df_chl["chl"])

df_chl.hvplot.points(
    x="lon", y="lat",
    c="log_chl",
    colormap="YlGn",
    clim=(df_chl["log_chl"].quantile(0.02), df_chl["log_chl"].quantile(0.98)),
    clabel="log₁₀ CHL (mg/m³)",
    title=f"Sentinel-3 OLCI Chlorophyll-a — Gulf of Tunis ({str(item_chl.datetime.date())})",
    xlabel="Longitude", ylabel="Latitude",
    size=6,
    geo=True,
    tiles="OSM",
    width=700, height=500,
)

---
## 4. Summary

| Variable | Source | Type | Resolution | Date |
|----------|--------|------|------------|------|
| SST | CMEMS L3S NRT | Gridded (regular) | ~1 km | 2025-10-01 |
| SST | Sentinel-3A SLSTR L2P | Swath (irregular) | ~1 km | varies |
| Chlorophyll-a | Sentinel-3A OLCI WFR L2 | Swath (irregular) | ~300 m | varies |

**Key points:**
- The CMEMS product is a gridded, gap-filled composite — convenient for time series and spatial comparisons, but smoothed.
- The Planetary Computer STAC products are raw Level-2 swaths — higher spatial resolution but cloudy pixels are missing and the satellite only passes every ~2 days.
- Chlorophyll-a is displayed on a **log₁₀ scale** because its natural variability spans several orders of magnitude (0.01 – 10 mg/m³ in coastal seas).
- The Gulf of Tunis is a semi-enclosed, relatively shallow sea influenced by the Medjerda River discharge in the south — this drives elevated turbidity and chlorophyll near the coast.

**Next steps:**
- Add in-situ profile data from the Copernicus Marine in-situ service to validate satellite SST
- Extend the chlorophyll search to multiple passes and build a time series
- Overlay surface currents from a CMEMS physics model to explain spatial patterns